In [0]:
cbsl_telecom_df = spark.table(
    "workspace.census360.gold_cbsl_telecom_analytical_v1"
)

cbsl_hies_df = spark.table(
    "workspace.census360.gold_cbsl_hies_analytical_v1"
)

cbsl_poverty_national_df = spark.table(
    "workspace.census360.gold_cbsl_poverty_national_analytical_v1"
)

cbsl_poverty_province_df = spark.table(
    "workspace.census360.gold_cbsl_poverty_province_analytical_v1"
)

cbsl_provincial_socioeconomic_df = spark.table(
    "workspace.census360.gold_cbsl_provincial_socioeconomic_analytical_v1"
)

print("All five CBSL Gold tables loaded successfully.")

In [0]:
dataframes = {
    "Telecom": cbsl_telecom_df,
    "HIES": cbsl_hies_df,
    "National poverty": cbsl_poverty_national_df,
    "Provincial poverty": cbsl_poverty_province_df,
    "Provincial socioeconomic": cbsl_provincial_socioeconomic_df
}

for name, df in dataframes.items():
    print(f"{name}: {df.count()} rows, {len(df.columns)} columns")

In [0]:
for name, df in dataframes.items():
    print(f"\n{name} columns:")
    print(df.columns)

In [0]:
for name, df in dataframes.items():
    print(f"\n{name} schema:")
    df.printSchema()

In [0]:
print("Telecom years:")
cbsl_telecom_df.select("year").distinct().orderBy("year").show()

print("HIES periods:")
cbsl_hies_df.select(
    "survey_period",
    "representative_year"
).distinct().orderBy("representative_year").show()

print("National poverty periods:")
cbsl_poverty_national_df.select(
    "survey_period",
    "representative_year",
    "definition_variant"
).distinct().orderBy("representative_year").show()

print("Provincial poverty geography types:")
cbsl_poverty_province_df.select(
    "geography_type"
).distinct().show()

print("Provincial socioeconomic geography types:")
cbsl_provincial_socioeconomic_df.select(
    "geography_type"
).distinct().show()

In [0]:
duplicate_checks = {
    "Telecom": (
        cbsl_telecom_df,
        ["year"]
    ),
    "HIES": (
        cbsl_hies_df,
        ["survey_period"]
    ),
    "National poverty": (
        cbsl_poverty_national_df,
        ["survey_period", "definition_variant"]
    ),
    "Provincial poverty": (
        cbsl_poverty_province_df,
        ["geography_name", "survey_period", "definition_variant"]
    ),
    "Provincial socioeconomic": (
        cbsl_provincial_socioeconomic_df,
        ["geography_name", "analysis_year"]
    )
}

for name, (df, key_columns) in duplicate_checks.items():
    duplicate_count = (
        df.groupBy(*key_columns)
        .count()
        .filter("count > 1")
        .count()
    )

    print(f"{name}: {duplicate_count} duplicate key groups")

In [0]:
from pyspark.sql import functions as F

for name, df in dataframes.items():
    print(f"\n{name} null counts:")

    null_counts = df.select([
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in df.columns
    ])

    null_counts.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

summary_rows = [
    (
        "Telecom",
        "national",
        cbsl_telecom_df.count(),
        cbsl_telecom_df.select("year").distinct().count(),
        cbsl_telecom_df.agg(F.min("year")).first()[0],
        cbsl_telecom_df.agg(F.max("year")).first()[0],
        cbsl_telecom_df.filter(
            F.col("internet_connections_yoy_growth_percent").isNotNull()
        ).count(),
        "Annual data; lag and growth nulls are expected in early years."
    ),
    (
        "HIES",
        "national",
        cbsl_hies_df.count(),
        cbsl_hies_df.select("representative_year").distinct().count(),
        cbsl_hies_df.agg(F.min("representative_year")).first()[0],
        cbsl_hies_df.agg(F.max("representative_year")).first()[0],
        cbsl_hies_df.filter(
            F.col("mean_household_income_change_percent").isNotNull()
        ).count(),
        "Irregular survey periods; no interpolation; only 2013, 2016 and 2019 overlap telecom."
    ),
    (
        "National poverty",
        "national",
        cbsl_poverty_national_df.count(),
        cbsl_poverty_national_df.select("representative_year").distinct().count(),
        cbsl_poverty_national_df.agg(F.min("representative_year")).first()[0],
        cbsl_poverty_national_df.agg(F.max("representative_year")).first()[0],
        cbsl_poverty_national_df.filter(
            F.col("poverty_headcount_change_pp").isNotNull()
        ).count(),
        "2019(a) and 2019(b) use different poverty definitions."
    ),
    (
        "Provincial poverty",
        "province",
        cbsl_poverty_province_df.count(),
        cbsl_poverty_province_df.select("representative_year").distinct().count(),
        cbsl_poverty_province_df.agg(F.min("representative_year")).first()[0],
        cbsl_poverty_province_df.agg(F.max("representative_year")).first()[0],
        cbsl_poverty_province_df.filter(
            F.col("poverty_headcount_change_pp").isNotNull()
        ).count(),
        "Use 2016 and 2019(a) for comparable province analysis; keep 2019(b) separate."
    ),
    (
        "Provincial socioeconomic",
        "province",
        cbsl_provincial_socioeconomic_df.count(),
        cbsl_provincial_socioeconomic_df.select("analysis_year").distinct().count(),
        cbsl_provincial_socioeconomic_df.agg(F.min("analysis_year")).first()[0],
        cbsl_provincial_socioeconomic_df.agg(F.max("analysis_year")).first()[0],
        cbsl_provincial_socioeconomic_df.filter(
            F.col("mean_household_income_change_percent").isNotNull()
        ).count(),
        "Only 2016 and 2019; income and expenditure changes are nominal."
    )
]

cbsl_observation_availability_df = spark.createDataFrame(
    summary_rows,
    [
        "dataset_name",
        "geography_level",
        "observation_count",
        "distinct_time_count",
        "minimum_year",
        "maximum_year",
        "usable_change_observations",
        "important_limitations"
    ]
)

cbsl_observation_availability_df.show(truncate=False)

In [0]:
telecom_hies_overlap_years_df = (
    cbsl_telecom_df
    .selectExpr("year as overlap_year")
    .distinct()
    .join(
        cbsl_hies_df
        .selectExpr("representative_year as overlap_year")
        .distinct(),
        on="overlap_year",
        how="inner"
    )
    .orderBy("overlap_year")
)

print("Telecom–HIES overlapping years:")
telecom_hies_overlap_years_df.show()

In [0]:
telecom_poverty_overlap_df = (
    cbsl_telecom_df
    .selectExpr("year as overlap_year")
    .distinct()
    .join(
        cbsl_poverty_national_df.select(
            F.col("representative_year").alias("overlap_year"),
            "survey_period",
            "definition_variant"
        ),
        on="overlap_year",
        how="inner"
    )
    .orderBy("overlap_year", "survey_period")
)

telecom_poverty_overlap_df.show(truncate=False)

In [0]:
print("Provincial poverty names:")

cbsl_poverty_province_df.filter(
    F.col("survey_period").isin("2016", "2019(a)")
).select(
    "geography_name"
).distinct().orderBy(
    "geography_name"
).show(truncate=False)


print("Provincial socioeconomic names:")

cbsl_provincial_socioeconomic_df.select(
    "geography_name"
).distinct().orderBy(
    "geography_name"
).show(truncate=False)

In [0]:
poverty_comparable_df = (
    cbsl_poverty_province_df
    .filter(F.col("survey_period").isin("2016", "2019(a)"))
    .select(
        "geography_name",
        F.col("representative_year").alias("analysis_year"),
        "survey_period",
        "poverty_headcount_index_percent",
        "poverty_gap_index_percent",
        "poverty_headcount_change_pp"
    )
)

cbsl_province_socioeconomic_poverty_joined_df = (
    cbsl_provincial_socioeconomic_df
    .join(
        poverty_comparable_df,
        on=["geography_name", "analysis_year"],
        how="inner"
    )
)

print("Socioeconomic rows:", cbsl_provincial_socioeconomic_df.count())
print("Comparable poverty rows:", poverty_comparable_df.count())
print("Joined rows:", cbsl_province_socioeconomic_poverty_joined_df.count())

cbsl_province_socioeconomic_poverty_joined_df.select(
    "geography_name",
    "analysis_year",
    "survey_period",
    "mean_income_per_household_monthly_lkr",
    "poverty_headcount_index_percent"
).orderBy(
    "analysis_year",
    "geography_name"
).show(20, truncate=False)

In [0]:
socio_keys_df = cbsl_provincial_socioeconomic_df.select(
    "geography_name",
    "analysis_year"
)

poverty_keys_df = poverty_comparable_df.select(
    "geography_name",
    "analysis_year"
)

unmatched_socio = socio_keys_df.join(
    poverty_keys_df,
    ["geography_name", "analysis_year"],
    "left_anti"
).count()

unmatched_poverty = poverty_keys_df.join(
    socio_keys_df,
    ["geography_name", "analysis_year"],
    "left_anti"
).count()

duplicate_join_keys = (
    cbsl_province_socioeconomic_poverty_joined_df
    .groupBy("geography_name", "analysis_year")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

critical_null_rows = (
    cbsl_province_socioeconomic_poverty_joined_df
    .filter(
        F.col("mean_income_per_household_monthly_lkr").isNull()
        | F.col("median_income_per_household_monthly_lkr").isNull()
        | F.col("household_expenditure_monthly_lkr").isNull()
        | F.col("poverty_headcount_index_percent").isNull()
    )
    .count()
)

print("Unmatched socioeconomic keys:", unmatched_socio)
print("Unmatched poverty keys:", unmatched_poverty)
print("Duplicate joined key groups:", duplicate_join_keys)
print("Rows with critical nulls:", critical_null_rows)

In [0]:
association_features = [
    "mean_income_per_household_monthly_lkr",
    "median_income_per_household_monthly_lkr",
    "mean_income_per_person_monthly_lkr",
    "household_expenditure_monthly_lkr",
    "expenditure_share_communication_percent",
    "education_passed_gce_ol_percent",
    "education_passed_gce_al_and_above_percent",
    "gini_coefficient_households"
]

correlation_rows = []

for period_label, period_df in [
    ("All observations", cbsl_province_socioeconomic_poverty_joined_df),
    ("2016", cbsl_province_socioeconomic_poverty_joined_df.filter(F.col("analysis_year") == 2016)),
    ("2019", cbsl_province_socioeconomic_poverty_joined_df.filter(F.col("analysis_year") == 2019))
]:
    result = period_df.select([
        F.corr(
            F.col(feature),
            F.col("poverty_headcount_index_percent")
        ).alias(feature)
        for feature in association_features
    ]).first()

    for feature in association_features:
        correlation_rows.append(
            (period_label, feature, round(result[feature], 4))
        )

cbsl_province_pearson_correlations_df = spark.createDataFrame(
    correlation_rows,
    ["analysis_period", "socioeconomic_indicator", "pearson_correlation"]
)

cbsl_province_pearson_correlations_df.orderBy(
    "analysis_period",
    F.abs(F.col("pearson_correlation")).desc()
).show(30, truncate=False)

In [0]:
cbsl_province_change_association_df = (
    cbsl_province_socioeconomic_poverty_joined_df
    .filter(
        (F.col("analysis_year") == 2019)
        & F.col("poverty_headcount_change_pp").isNotNull()
    )
)

change_features = [
    "mean_household_income_change_percent",
    "mean_per_person_income_change_percent",
    "median_household_income_change_percent",
    "household_expenditure_change_percent",
    "communication_expenditure_share_change_pp",
    "education_expenditure_share_change_pp",
    "household_gini_change_points"
]

change_correlations = []

for feature in change_features:
    value = cbsl_province_change_association_df.stat.corr(
        feature,
        "poverty_headcount_change_pp"
    )

    change_correlations.append(
        (feature, round(value, 4))
    )

cbsl_change_correlations_df = spark.createDataFrame(
    change_correlations,
    ["change_indicator", "pearson_correlation_with_poverty_change"]
)

print("Usable province change observations:",
      cbsl_province_change_association_df.count())

cbsl_change_correlations_df.orderBy(
    F.abs(F.col("pearson_correlation_with_poverty_change")).desc()
).show(truncate=False)

In [0]:
province_rankings_df = (
    cbsl_province_socioeconomic_poverty_joined_df
    .filter(F.col("analysis_year") == 2019)
    .select(
        "geography_name",
        "poverty_headcount_index_percent",
        "mean_income_per_household_monthly_lkr",
        "mean_income_per_person_monthly_lkr",
        "household_expenditure_monthly_lkr",
        "education_passed_gce_al_and_above_percent",
        "expenditure_share_communication_percent"
    )
    .orderBy(
        F.col("poverty_headcount_index_percent").desc()
    )
)

province_rankings_df.show(truncate=False)

In [0]:
level_observations = cbsl_province_socioeconomic_poverty_joined_df.count()
change_observations = cbsl_province_change_association_df.count()

level_predictors = 8
change_predictors = 7

print("Province-level observations:", level_observations)
print("Province-level candidate predictors:", level_predictors)
print("Change observations:", change_observations)
print("Change candidate predictors:", change_predictors)

print("\nModelling decision:")
print("- National datasets: too few overlapping observations for reliable modelling.")
print("- Province-level data: only 18 level observations and 9 change observations.")
print("- The sample is too small relative to the number of candidate predictors.")
print("- Descriptive association analysis is more defensible than regression or machine learning.")

In [0]:
cbsl_national_telecom_hies_joined_df = (
    cbsl_telecom_df
    .join(
        cbsl_hies_df,
        cbsl_telecom_df.year == cbsl_hies_df.representative_year,
        "inner"
    )
)

cbsl_national_telecom_poverty_joined_df = (
    cbsl_telecom_df
    .join(
        cbsl_poverty_national_df,
        cbsl_telecom_df.year == cbsl_poverty_national_df.representative_year,
        "inner"
    )
)

print(
    "Telecom–HIES joined rows:",
    cbsl_national_telecom_hies_joined_df.count()
)

print(
    "Telecom–poverty joined rows:",
    cbsl_national_telecom_poverty_joined_df.count()
)

print("\nTelecom–poverty observations:")
cbsl_national_telecom_poverty_joined_df.select(
    "year",
    "survey_period",
    "definition_variant",
    "poverty_headcount_index_percent"
).orderBy(
    "year",
    "survey_period"
).show(truncate=False)

In [0]:
from pyspark.sql.window import Window

income_window = Window.orderBy(
    F.col("mean_income_per_household_monthly_lkr").desc()
)

poverty_window = Window.orderBy(
    F.col("poverty_headcount_index_percent").asc()
)

cbsl_province_rank_gap_df = (
    cbsl_province_socioeconomic_poverty_joined_df
    .filter(F.col("analysis_year") == 2019)
    .select(
        "geography_name",
        "mean_income_per_household_monthly_lkr",
        "poverty_headcount_index_percent"
    )
    .withColumn(
        "income_rank_highest_first",
        F.row_number().over(income_window)
    )
    .withColumn(
        "poverty_rank_lowest_first",
        F.row_number().over(poverty_window)
    )
    .withColumn(
        "rank_gap",
        F.abs(
            F.col("income_rank_highest_first")
            - F.col("poverty_rank_lowest_first")
        )
    )
    .orderBy(F.col("rank_gap").desc(), "geography_name")
)

cbsl_province_rank_gap_df.show(truncate=False)

In [0]:
strongest_level = (
    cbsl_province_pearson_correlations_df
    .filter(F.col("analysis_period") == "All observations")
    .orderBy(F.abs(F.col("pearson_correlation")).desc())
    .first()
)

strongest_change = (
    cbsl_change_correlations_df
    .orderBy(
        F.abs(
            F.col("pearson_correlation_with_poverty_change")
        ).desc()
    )
    .first()
)

highest_poverty_2019 = (
    cbsl_province_socioeconomic_poverty_joined_df
    .filter(F.col("analysis_year") == 2019)
    .orderBy(F.col("poverty_headcount_index_percent").desc())
    .first()
)

largest_rank_gap = cbsl_province_rank_gap_df.first()

summary_rows = [
    (
        "National Telecom–HIES overlap",
        "3 observations: 2013, 2016 and 2019",
        "Descriptive comparison only"
    ),
    (
        "National Telecom–poverty overlap",
        "4 rows, including separate 2019(a) and 2019(b) definitions",
        "Do not combine poverty definitions"
    ),
    (
        "Strongest province-level association",
        f"{strongest_level['socioeconomic_indicator']}: "
        f"{strongest_level['pearson_correlation']}",
        "Exploratory association; not causal"
    ),
    (
        "Strongest province-change association",
        f"{strongest_change['change_indicator']}: "
        f"{strongest_change['pearson_correlation_with_poverty_change']}",
        "Only 9 observations"
    ),
    (
        "Highest poverty province in 2019(a)",
        f"{highest_poverty_2019['geography_name']}: "
        f"{highest_poverty_2019['poverty_headcount_index_percent']}%",
        "Old-line comparable poverty definition"
    ),
    (
        "Largest income–poverty rank gap",
        f"{largest_rank_gap['geography_name']}: "
        f"rank gap {largest_rank_gap['rank_gap']}",
        "Possible unusual provincial pattern"
    ),
    (
        "Modelling decision",
        "Regression and machine learning are not justified",
        "Sample size is too small"
    )
]

cbsl_final_analysis_summary_df = spark.createDataFrame(
    summary_rows,
    ["analysis_area", "result", "interpretation_note"]
)

cbsl_final_analysis_summary_df.show(truncate=False)

In [0]:
province_join_output_df = (
    cbsl_province_socioeconomic_poverty_joined_df
    .select(
        "geography_name",
        "analysis_year",
        "survey_period",
        "mean_income_per_household_monthly_lkr",
        "mean_income_per_person_monthly_lkr",
        "median_income_per_household_monthly_lkr",
        "household_expenditure_monthly_lkr",
        "expenditure_share_communication_percent",
        "education_passed_gce_ol_percent",
        "education_passed_gce_al_and_above_percent",
        "gini_coefficient_households",
        "poverty_headcount_index_percent",
        "poverty_gap_index_percent",
        "poverty_headcount_change_pp"
    )
)

association_summary_output_df = (
    cbsl_province_pearson_correlations_df
    .withColumn("association_type", F.lit("level"))
    .withColumnRenamed("analysis_period", "analysis_group")
    .withColumnRenamed("socioeconomic_indicator", "indicator")
    .withColumnRenamed("pearson_correlation", "correlation")
    .select("association_type", "analysis_group", "indicator", "correlation")
    .unionByName(
        cbsl_change_correlations_df
        .withColumn("association_type", F.lit("change"))
        .withColumn("analysis_group", F.lit("2016_to_2019"))
        .withColumnRenamed("change_indicator", "indicator")
        .withColumnRenamed(
            "pearson_correlation_with_poverty_change",
            "correlation"
        )
        .select("association_type", "analysis_group", "indicator", "correlation")
    )
)

national_overlap_output_df = cbsl_final_analysis_summary_df.filter(
    F.col("analysis_area").startswith("National")
)

outputs = {
    "Province joined output": province_join_output_df,
    "Association summary output": association_summary_output_df,
    "National overlap output": national_overlap_output_df
}

for name, df in outputs.items():
    print(f"\n{name}: {df.count()} rows, {len(df.columns)} columns")
    print(df.columns)

In [0]:
target_tables = [
    "workspace.census360.gold_cbsl_province_poverty_socioeconomic_joined_v1",
    "workspace.census360.gold_cbsl_province_association_summary_v1",
    "workspace.census360.gold_cbsl_national_overlap_summary_v1"
]

for table_name in target_tables:
    print(
        table_name,
        "Exists:" if spark.catalog.tableExists(table_name) else "Available:"
    )

In [0]:
tables_to_save = {
    "workspace.census360.gold_cbsl_province_poverty_socioeconomic_joined_v1":
        province_join_output_df,

    "workspace.census360.gold_cbsl_province_association_summary_v1":
        association_summary_output_df,

    "workspace.census360.gold_cbsl_national_overlap_summary_v1":
        national_overlap_output_df
}

for target_table, dataframe in tables_to_save.items():

    if spark.catalog.tableExists(target_table):
        print(f"Table already exists. Nothing was written: {target_table}")

    else:
        (
            dataframe.write
            .format("delta")
            .mode("append")
            .saveAsTable(target_table)
        )

        print(f"Table created successfully: {target_table}")

In [0]:
final_tables = [
    "workspace.census360.gold_cbsl_province_poverty_socioeconomic_joined_v1",
    "workspace.census360.gold_cbsl_province_association_summary_v1",
    "workspace.census360.gold_cbsl_national_overlap_summary_v1"
]

for table_name in final_tables:
    df = spark.table(table_name)

    print(f"Table: {table_name}")
    print(f"Exists: {spark.catalog.tableExists(table_name)}")
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print("-" * 70)